# ADV-02: Advanced Digital Image Processing & Mathematical Inverse Problems
## Rudin-Osher-Fatemi (ROF) Total Variation Regularization & ADMM Solver

**Author:** Aji Pamungkas Tri Nurcahyo (Institut Teknologi Sumatera - ITERA)  
**Level:** L3 Advanced Curriculum  
**Pillar:** Pillar 2 — Intelligent Visual Perception & Sensing (Roadmap ISVWS 2026–2040)  

---
### Abstract
This interactive notebook provides a complete mathematical and computational implementation of variational image restoration:
1. **Forward Degradation Model**: $y = A x + n$, simulating spatial blur (PSF) and measurement noise.
2. **Ill-Posedness Analysis**: Instability of naive unregularized inverse filtering due to singular value decay $\sigma_i(A) \to 0$.
3. **Rudin-Osher-Fatemi (ROF) Variational Model**: $\min_x \frac{1}{2} \|Ax - y\|_2^2 + \lambda \|\nabla x\|_1$.
4. **ADMM Variable Splitting**: Alternating direction optimization using 2D FFT for $x$-update and soft-thresholding $\mathcal{S}_{\lambda/\rho}$ for $z$-update.
5. **Convergence Telemetry**: Monitoring primal residual $\|r^k\|_F$ and dual residual $\|s^k\|_F$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy Version: {np.__version__}")

### 1. Synthetic Ground Truth Generation
We generate a normalized $64 \times 64$ image $x \in [0, 1]$ containing sharp step discontinuities (piecewise constant model) to evaluate edge-preservation.

In [ ]:
np.random.seed(42)
H, W = 64, 64
clean = np.zeros((H, W), dtype=float)
clean[:, W // 2 :] = 0.85
clean[20:44, 20:44] = 0.45

def compute_tv(img):
    dx = np.roll(img, -1, axis=1) - img
    dy = np.roll(img, -1, axis=0) - img
    return float(np.sum(np.sqrt(dx**2 + dy**2)))

print(f"Clean Image TV: {compute_tv(clean):.2f}")
plt.figure(figsize=(4, 4))
plt.imshow(clean, cmap='gray', vmin=0, vmax=1)
plt.title("Ground Truth Clean Image (x)")
plt.axis('off')
plt.show()

### 2. Forward Degradation Model: $y = Ax + n$
We apply a $3 \times 3$ Gaussian blur kernel $A$ (Optical Transfer Function) and additive Gaussian noise $n \sim \mathcal{N}(0, \sigma^2)$.

In [ ]:
k_size = 3
kernel = np.array([[1/16, 2/16, 1/16], [2/16, 4/16, 2/16], [1/16, 2/16, 1/16]], dtype=float)
psf_padded = np.zeros((H, W), dtype=float)
psf_padded[:k_size, :k_size] = kernel
psf_zero_phase = np.roll(np.roll(psf_padded, -(k_size // 2), axis=0), -(k_size // 2), axis=1)
otf_A = np.fft.fft2(psf_zero_phase)

blurred = np.real(np.fft.ifft2(np.fft.fft2(clean) * otf_A))
noise_sigma = 0.05
noise = np.random.normal(0.0, noise_sigma, size=clean.shape)
degraded = np.clip(blurred + noise, 0.0, 1.0)

def compute_psnr(ref, tgt):
    mse = np.mean((ref - tgt)**2)
    return 20.0 * np.log10(1.0 / np.sqrt(max(1e-12, mse)))

print(f"Degraded PSNR: {compute_psnr(clean, degraded):.2f} dB")
print(f"Degraded TV: {compute_tv(degraded):.2f}")

### 3. ADMM Rudin-Osher-Fatemi (ROF) Variational Solver
We solve:
$$\min_{x, z} \frac{1}{2}\|Ax - y\|_2^2 + \lambda \|z\|_1 \quad \text{s.t.} \quad z = \nabla x$$
using alternating minimization of the Augmented Lagrangian.

In [ ]:
def soft_threshold(v, tau):
    return np.sign(v) * np.maximum(np.abs(v) - tau, 0.0)

# Periodic difference operators
dx = np.zeros((H, W), dtype=float); dx[0, 0] = -1.0; dx[0, W - 1] = 1.0
dy = np.zeros((H, W), dtype=float); dy[0, 0] = -1.0; dy[H - 1, 0] = 1.0
otf_Dx = np.fft.fft2(dx)
otf_Dy = np.fft.fft2(dy)

lam = 0.015
rho = 0.15
max_iter = 35

denom = np.abs(otf_A)**2 + rho * (np.abs(otf_Dx)**2 + np.abs(otf_Dy)**2)
x = np.copy(degraded)
zx = np.zeros((H, W), dtype=float)
zy = np.zeros((H, W), dtype=float)
ux = np.zeros((H, W), dtype=float)
uy = np.zeros((H, W), dtype=float)

primal_res = []
dual_res = []

for it in range(max_iter):
    prev_zx = np.copy(zx)
    prev_zy = np.copy(zy)
    
    # x-update via 2D FFT
    num = np.conj(otf_A) * np.fft.fft2(degraded) + rho * (
        np.conj(otf_Dx) * np.fft.fft2(zx - ux) + np.conj(otf_Dy) * np.fft.fft2(zy - uy)
    )
    x = np.clip(np.real(np.fft.ifft2(num / (denom + 1e-12))), 0.0, 1.0)
    
    # Gradient
    gx = np.roll(x, -1, axis=1) - x
    gy = np.roll(x, -1, axis=0) - x
    
    # z-update via proximal shrinkage
    tau = lam / rho
    zx = soft_threshold(gx + ux, tau)
    zy = soft_threshold(gy + uy, tau)
    
    # u-update
    ux += gx - zx
    uy += gy - zy
    
    r_norm = float(np.sqrt(np.sum((gx - zx)**2 + (gy - zy)**2)))
    s_norm = float(rho * np.sqrt(np.sum((zx - prev_zx)**2 + (zy - prev_zy)**2)))
    primal_res.append(r_norm)
    dual_res.append(s_norm)

psnr_rest = compute_psnr(clean, x)
print(f"Final Restored PSNR: {psnr_rest:.2f} dB (Gain: +{psnr_rest - compute_psnr(clean, degraded):.2f} dB)")
print(f"Restored TV: {compute_tv(x):.2f}")

### 4. Visual Comparison and Convergence Telemetry
We visualize the Ground Truth, Degraded, and Restored images alongside the error map and primal/dual residual convergence plot.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Ground Truth")
axes[0].axis('off')

axes[1].imshow(degraded, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f"Degraded\nPSNR: {compute_psnr(clean, degraded):.2f} dB")
axes[1].axis('off')

axes[2].imshow(x, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f"ADMM TV Restored\nPSNR: {psnr_rest:.2f} dB")
axes[2].axis('off')

axes[3].imshow(np.abs(clean - x), cmap='hot')
axes[3].set_title("Residual Error Map |x - x_hat|")
axes[3].axis('off')

plt.tight_layout()
plt.show()

# Residual convergence plot
plt.figure(figsize=(7, 3))
plt.plot(primal_res, label=r'Primal Residual ||r^k||_F', color='#38bdf8', lw=2)
plt.plot(dual_res, label=r'Dual Residual ||s^k||_F', color='#34d399', lw=2, ls='--')
plt.yscale('log')
plt.xlabel("Iteration k")
plt.ylabel("Residual Norm (log scale)")
plt.title("ADMM Convergence Telemetry")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()